# BX2S-Net — Inter-Comparison Baseline for PPCNet
## Adapted from: Chen et al., "BX2S-Net: Learning to Reconstruct 3D Spinal Structures from Bi-planar X-ray Images", Comput. Biol. Med. 2023

### Adaptation Protocol (identical to 3D-ReVert / X2CT-GAN inter-comparison)
| Aspect | What we keep from BX2S-Net | What we keep from PPCNet v12.3 |
|---|---|---|
| **Dimensionality Enhancement** | 2D AP/LP images expanded to pseudo-3D and concatenated | — |
| **Encoder** | VGG-style 3D encoder with BatchNorm (dimensionally-consistent) | — |
| **Decoder** | 3D decoder with FFAG (Full-scale Feature Attention Guidance) | — |
| **Loss** | Spatially-weighted BCE + Dice (boundary-enhanced) | — |
| **Optimizer** | Adam lr=1e-3, weight decay=1e-4 | — |
| **Dataset** | — | Lumbar_Filtered_1037 (829/103/105) |
| **Coordinate system** | — | pts_to_local / local_to_world / compute_scale |
| **GT representation** | Volume (occupancy at 64 cubed) from GT point cloud | GT point cloud for eval |
| **Evaluation** | — | Chamfer, F@1/2/5 mm, HD95 (world-mm) |
| **N_POINTS** | — | 8192 |

### Key architectural differences from PPCNet v12.3
- **Output**: 3D occupancy volume (64 cubed) then extract point cloud at test time (vs direct point cloud)
- **Input**: Dimensionally-enhanced pseudo-3D tensor (vs separate 2D feature maps)
- **Encoder**: VGG-style 3D convolutions with BatchNorm (vs ResNet-34 2D with BatchNorm)
- **Decoder**: FFAG multi-scale attention at every level (vs iterative MLP refinement)
- **Loss**: Spatially-weighted BCE + Dice (vs Chamfer + gap + axial + extent + curvature + SW)
- **No projection matrices used**: BX2S-Net does not use calibrated P matrices
- **No iterative refinement**: single forward pass through 3D encoder-decoder


In [ ]:
"""
BX2S-Net Inter-Comparison Baseline
====================================
Chen et al., Computers in Biology and Medicine 2023
Architecture: Dimensionality Enhancement + 3D VGG Encoder + FFAG Decoder -> 64^3 volume
Loss: Spatially-weighted BCE + Dice (their original)
Evaluation: Chamfer, F@1/2/5mm, HD95 in world-mm (our standard metrics)
"""
import os, sys, json, time, random, warnings, csv, math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import vtk
from tqdm import tqdm
from scipy import ndimage

warnings.filterwarnings("ignore", category=FutureWarning)

# -- GPU MEMORY CAP (MIG-aware) --
# On a MIG partition total_memory reports the slice size (~9.7GB), not the full
# physical GPU (80GB). The old code computed 50.0/total_gb which blew past 1.0
# and got clamped to 0.95 -- leaving zero safety margin on a small slice.
# Fix: target a concrete GB value, auto-capped to 90% of whatever is visible.
TARGET_GPU_MEM_GB = 30
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:512")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Visible device memory (MIG slice or full GPU): {total_gb:.2f} GB")
    frac = min(TARGET_GPU_MEM_GB / total_gb, 0.90)
    if TARGET_GPU_MEM_GB > total_gb:
        print(f"  WARNING: requested {TARGET_GPU_MEM_GB:.1f}GB > visible {total_gb:.2f}GB. "
              f"Capping to {frac*total_gb:.2f}GB (90% of visible device).")
    torch.cuda.set_per_process_memory_fraction(frac, 0)
    print(f"  Memory fraction set: {frac:.3f} (~{frac*total_gb:.2f} GB)")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# -- PATHS --
DATA_ROOT   = Path("/path/to/Lumbar_Filtered_1037")
SPLIT_FILE  = DATA_ROOT / "dataset_split.json"
PROJECT_DIR = Path("/path/to/inter_comparison_bx2snet")
CKPT_DIR    = PROJECT_DIR / "checkpoints"
LOG_DIR     = PROJECT_DIR / "logs"
RESULTS_DIR = PROJECT_DIR / "results"
for d in [PROJECT_DIR, CKPT_DIR, LOG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# -- MODEL CONFIG (BX2S-Net settings) --
IMG_SIZE        = 64            # resized to match volume dim for dimensionality enhancement
VOL_SIZE        = 64            # 3D output volume resolution
N_POINTS        = 8192          # point cloud size for evaluation (same as v12.3)
ENC_CHANNELS    = [32, 64, 128, 256]   # 3D encoder channel progression
DEC_CHANNELS    = [256, 128, 64, 32]   # 3D decoder channel progression

# -- TRAINING CONFIG (BX2S-Net settings) --
BATCH_SIZE      = 4            # BX2S-Net: small batch for 3D volumes
NUM_WORKERS     = 4
EPOCHS          = 200           # BX2S-Net: 200 epochs
LR              = 1e-3          # BX2S-Net: Adam lr=1e-3
WEIGHT_DECAY    = 1e-4
GRAD_CLIP       = 1.0
EARLY_STOP_PATIENCE = 60        # same patience as our framework

# -- LOSS WEIGHTS (BX2S-Net original) --
LAMBDA_BCE      = 1.0           # spatially-weighted BCE
LAMBDA_DICE     = 1.0           # Dice loss
BOUNDARY_WEIGHT = 5.0           # extra weight on boundary voxels

MAX_EVAL        = 103
CKPT_PATH       = CKPT_DIR / "latest_checkpoint.pth"
BEST_CKPT       = CKPT_DIR / "best_checkpoint.pth"
TRAINING_LOG    = LOG_DIR  / "training_log.json"

print("=" * 65)
print("BX2S-Net Inter-Comparison -- Lumbar Spine Point Cloud")
print("=" * 65)
for k, v in dict(
    PAPER="Chen et al., Comput. Biol. Med. 2023",
    INPUT=f"Dimensionality-enhanced pseudo-3D from 2 x {IMG_SIZE}x{IMG_SIZE} DRR",
    OUTPUT=f"{VOL_SIZE}^3 occupancy volume -> {N_POINTS}-point cloud",
    LOSS=f"Spatially-weighted BCE(w={LAMBDA_BCE}) + Dice(w={LAMBDA_DICE})",
    OPTIMIZER=f"Adam (lr={LR}, wd={WEIGHT_DECAY})",
    EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE,
    EVAL_METRICS="Chamfer, F@1/2/5mm, HD95 (world-mm)",
).items():
    print(f"  {k:<18} = {v}")


Device : cuda
GPU    : NVIDIA A100-SXM4-80GB
Visible device memory (MIG slice or full GPU): 84.99 GB
  Memory fraction set: 0.353 (~30.00 GB)
BX2S-Net Inter-Comparison -- Lumbar Spine Point Cloud
  PAPER              = Chen et al., Comput. Biol. Med. 2023
  INPUT              = Dimensionality-enhanced pseudo-3D from 2 x 64x64 DRR
  OUTPUT             = 64^3 occupancy volume -> 8192-point cloud
  LOSS               = Spatially-weighted BCE(w=1.0) + Dice(w=1.0)
  OPTIMIZER          = Adam (lr=0.001, wd=0.0001)
  EPOCHS             = 200
  BATCH_SIZE         = 4
  EVAL_METRICS       = Chamfer, F@1/2/5mm, HD95 (world-mm)


In [2]:
# ==============================================================================
# DATA UTILITIES — same coordinate system as PPCNet v12.3
# ==============================================================================

def load_vtk_points(path):
    r = vtk.vtkPolyDataReader(); r.SetFileName(str(path)); r.Update()
    p = r.GetOutput()
    return np.array([p.GetPoint(i) for i in range(p.GetNumberOfPoints())], np.float32)

def save_vtk_points(points, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    vp = vtk.vtkPoints()
    for pt in points: vp.InsertNextPoint(float(pt[0]), float(pt[1]), float(pt[2]))
    vc = vtk.vtkCellArray()
    for i in range(len(points)): vc.InsertNextCell(1); vc.InsertCellPoint(i)
    pd = vtk.vtkPolyData(); pd.SetPoints(vp); pd.SetVerts(vc)
    w = vtk.vtkPolyDataWriter()
    w.SetFileName(str(path)); w.SetInputData(pd); w.SetFileTypeToASCII(); w.Write()

def load_drr_image(path, size=IMG_SIZE):
    """Load DRR and resize to BX2S-Net input resolution (64x64)."""
    img = Image.open(path).convert("L")
    if img.size != (size, size): img = img.resize((size, size), Image.BILINEAR)
    return np.array(img, np.float32) / 255.0

def load_split(p=SPLIT_FILE):
    with open(p) as f: d = json.load(f)
    return d["train"], d["val"], d["test"]

def append_log(path, rec):
    log = {"records": []}
    if path.exists():
        with open(path) as f: log = json.load(f)
    log["records"].append(rec)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w") as f: json.dump(log, f, indent=2)
    tmp.replace(path)

def pts_to_local(pts, c, s): return (pts - c[None]) / (s[None] + 1e-6)
def local_to_world(pts, c, s): return pts * s[None] + c[None]

def compute_scale(gt):
    e = (gt.max(0) - gt.min(0)).astype(np.float32)
    s = e * 0.55 + np.array([20., 20., 30.], np.float32)
    return np.maximum(s, np.array([50., 50., 80.], np.float32))


def make_gt_volume(pts_local, vol_size=VOL_SIZE, dilation=1):
    """Convert GT point cloud (local [-1,1]^3) to binary occupancy volume."""
    p = np.clip((np.asarray(pts_local, np.float32) + 1) * 0.5, 0, 0.999999)
    idx = np.clip(np.floor(p * vol_size).astype(np.int32), 0, vol_size - 1)
    vol = np.zeros((vol_size,) * 3, np.float32)
    for dx in range(-dilation, dilation + 1):
        for dy in range(-dilation, dilation + 1):
            for dz in range(-dilation, dilation + 1):
                vol[np.clip(idx[:, 0] + dx, 0, vol_size - 1),
                    np.clip(idx[:, 1] + dy, 0, vol_size - 1),
                    np.clip(idx[:, 2] + dz, 0, vol_size - 1)] = 1.0
    return vol


def make_boundary_weight(vol, boundary_w=BOUNDARY_WEIGHT):
    """
    BX2S-Net spatially-weighted map: higher weight on boundary voxels.
    Boundary = dilated - eroded occupancy (surface voxels).
    """
    struct = ndimage.generate_binary_structure(3, 1)
    dilated = ndimage.binary_dilation(vol > 0.5, structure=struct, iterations=1).astype(np.float32)
    eroded  = ndimage.binary_erosion(vol > 0.5, structure=struct, iterations=1).astype(np.float32)
    boundary = dilated - eroded
    weight = np.ones_like(vol)
    weight[boundary > 0.5] = boundary_w
    return weight


def volume_to_points(vol, threshold=0.5, n_points=N_POINTS):
    """Extract point cloud from predicted occupancy volume. Returns local [-1,1]^3."""
    mask = vol > threshold
    coords = np.argwhere(mask).astype(np.float32)
    if len(coords) == 0:
        return np.random.uniform(-0.5, 0.5, (n_points, 3)).astype(np.float32)
    local = (coords + 0.5) / vol.shape[0] * 2.0 - 1.0
    if len(local) >= n_points:
        idx = np.random.choice(len(local), n_points, replace=False)
        return local[idx]
    else:
        extra = n_points - len(local)
        pad_idx = np.random.choice(len(local), extra, replace=True)
        pad = local[pad_idx] + np.random.randn(extra, 3).astype(np.float32) * 0.01
        return np.concatenate([local, pad], 0)


class LumbarDataset(Dataset):
    """
    Dataset for BX2S-Net inter-comparison.
    Returns dimensionality-enhanced pseudo-3D input, GT volume, boundary weight map,
    GT point cloud, center/scale.
    BX2S-Net does NOT use projection matrices.
    """
    def __init__(self, ids, root=DATA_ROOT, aug=False):
        self.ids = ids; self.root = Path(root); self.aug = aug

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        pid = self.ids[i]; d = self.root / pid

        # Load DRRs at 64x64 (to match VOL_SIZE for dimensionality enhancement)
        dap = load_drr_image(d / "AP_0"  / "drr_AP_0.png")
        dlp = load_drr_image(d / "LP_90" / "drr_LP_90.png")

        # Load GT point cloud
        gt = load_vtk_points(d / "gt_ppc.vtk")
        c  = gt.mean(0).astype(np.float32)
        s  = compute_scale(gt)
        gl = np.clip(pts_to_local(gt, c, s), -1, 1)

        # Create GT occupancy volume and boundary weight map
        gt_vol = make_gt_volume(gl)
        bw_map = make_boundary_weight(gt_vol)

        # Subsample GT for evaluation
        n = len(gt)
        sel = np.random.choice(n, N_POINTS, replace=(n < N_POINTS))

        # Simple augmentation: horizontal flip
        if self.aug and np.random.rand() < 0.5:
            dap = dap[:, ::-1].copy()
            dlp = dlp[:, ::-1].copy()
            gt_vol = gt_vol[::-1, :, :].copy()
            bw_map = bw_map[::-1, :, :].copy()

        # === BX2S-Net Dimensionality Enhancement ===
        # AP image (64x64) -> expand along sagittal axis -> (1, 64, 64, 64)
        ap_3d = np.tile(dap[np.newaxis, np.newaxis, :, :], (1, VOL_SIZE, 1, 1))  # (1, D, H, W)
        # LP image (64x64) -> expand along coronal axis -> (1, 64, 64, 64)
        lp_3d = np.tile(dlp[np.newaxis, :, np.newaxis, :], (1, 1, VOL_SIZE, 1))  # (1, H, D, W)
        # Concatenate -> (2, 64, 64, 64)
        input_3d = np.concatenate([ap_3d, lp_3d], axis=0)

        return {
            "input_3d":     torch.from_numpy(input_3d).float(),
            "gt_volume":    torch.from_numpy(gt_vol).float(),
            "boundary_wt":  torch.from_numpy(bw_map).float(),
            "gt_ppc_world": torch.from_numpy(gt[sel]).float(),
            "gt_ppc_local": torch.from_numpy(gl[sel]).float(),
            "center":       torch.from_numpy(c).float(),
            "scale":        torch.from_numpy(s).float(),
            "patient_id":   pid,
        }

train_ids, val_ids, test_ids = load_split()
print(f"Split: train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}")


Split: train=829  val=103  test=105


In [3]:
# ==============================================================================
# MODEL — BX2S-Net (Chen et al., Comput. Biol. Med. 2023)
# ==============================================================================
#
# Dimensionally-consistent 3D encoder-decoder with FFAG attention.
# Input: (B, 2, 64, 64, 64) — dimensionality-enhanced pseudo-3D from AP+LP
# Output: (B, 64, 64, 64) — binary occupancy volume
#
# Key components (faithful to paper):
#   1. VGG-style 3D encoder with BatchNorm (dimensionally consistent)
#   2. FFAG (Full-scale Feature Attention Guidance) at each decoder level
#   3. 3D decoder with transposed convolutions + skip connections
# ==============================================================================


class ConvBlock3D(nn.Module):
    """Double 3D convolution block: Conv-BN-ReLU x 2 (VGG-style)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


class ChannelAttention3D(nn.Module):
    """Channel attention: AdaptiveAvgPool + MLP squeeze-excitation."""
    def __init__(self, ch, reduction=8):
        super().__init__()
        self.att = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(ch, max(ch // reduction, 16)),
            nn.ReLU(inplace=True),
            nn.Linear(max(ch // reduction, 16), ch),
            nn.Sigmoid())

    def forward(self, x):
        w = self.att(x).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        return x * w


class SpatialAttention3D(nn.Module):
    """Spatial attention: Conv3D on channel-pooled features."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(2, 1, 7, padding=3, bias=False),
            nn.Sigmoid())

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.max(dim=1, keepdim=True)[0]
        att = self.conv(torch.cat([avg, mx], dim=1))
        return x * att


class FFAGModule(nn.Module):
    """
    Full-scale Feature Attention Guidance (FFAG) module.
    Aggregates features from ALL encoder levels at the current decoder spatial
    resolution, applies channel + spatial attention, and fuses with decoder features.
    This is the core novelty of BX2S-Net.
    """
    def __init__(self, enc_channels_list, dec_ch, spatial_size):
        super().__init__()
        self.spatial_size = spatial_size
        # 1x1x1 conv to project each encoder level to dec_ch
        self.projections = nn.ModuleList([
            nn.Sequential(
                nn.Conv3d(enc_ch, dec_ch, 1, bias=False),
                nn.BatchNorm3d(dec_ch),
                nn.ReLU(inplace=True))
            for enc_ch in enc_channels_list
        ])
        # Fuse all projected features (N_levels * dec_ch -> dec_ch)
        n_levels = len(enc_channels_list)
        self.fuse = nn.Sequential(
            nn.Conv3d(n_levels * dec_ch, dec_ch, 1, bias=False),
            nn.BatchNorm3d(dec_ch),
            nn.ReLU(inplace=True))
        # Channel + spatial attention
        self.ch_att = ChannelAttention3D(dec_ch)
        self.sp_att = SpatialAttention3D()

    def forward(self, enc_features_list):
        S = self.spatial_size
        projected = []
        for proj, feat in zip(self.projections, enc_features_list):
            # Resize all encoder features to current decoder spatial size
            if feat.shape[2:] != (S, S, S):
                feat = F.interpolate(feat, size=(S, S, S), mode='trilinear', align_corners=False)
            projected.append(proj(feat))
        # Concatenate and fuse
        cat = torch.cat(projected, dim=1)
        fused = self.fuse(cat)
        # Attention
        fused = self.ch_att(fused)
        fused = self.sp_att(fused)
        return fused


class BX2SEncoder(nn.Module):
    """
    VGG-style 3D encoder (dimensionally consistent).
    Input: (B, 2, 64, 64, 64)
    Output: 4-level feature pyramid + bottleneck.
    """
    def __init__(self):
        super().__init__()
        # Level 1: 2 -> 32, spatial 64 -> 32
        self.enc1 = ConvBlock3D(2, 32)
        self.pool1 = nn.MaxPool3d(2)
        # Level 2: 32 -> 64, spatial 32 -> 16
        self.enc2 = ConvBlock3D(32, 64)
        self.pool2 = nn.MaxPool3d(2)
        # Level 3: 64 -> 128, spatial 16 -> 8
        self.enc3 = ConvBlock3D(64, 128)
        self.pool3 = nn.MaxPool3d(2)
        # Level 4: 128 -> 256, spatial 8 -> 4
        self.enc4 = ConvBlock3D(128, 256)
        self.pool4 = nn.MaxPool3d(2)
        # Bottleneck at 4^3
        self.bottleneck = ConvBlock3D(256, 512)

    def forward(self, x):
        e1 = self.enc1(x);   p1 = self.pool1(e1)    # 32 @ 32^3
        e2 = self.enc2(p1);  p2 = self.pool2(e2)    # 64 @ 16^3
        e3 = self.enc3(p2);  p3 = self.pool3(e3)    # 128 @ 8^3
        e4 = self.enc4(p3);  p4 = self.pool4(e4)    # 256 @ 4^3
        bn = self.bottleneck(p4)                      # 512 @ 4^3
        return [e1, e2, e3, e4], bn


class BX2SDecoder(nn.Module):
    """
    3D decoder with FFAG attention at each level.
    Upsamples from 4^3 bottleneck to 64^3 output.
    """
    def __init__(self):
        super().__init__()
        enc_chs = [32, 64, 128, 256]

        # Upsample blocks (transposed convolution)
        self.up4 = nn.ConvTranspose3d(512, 256, 2, stride=2)      # 4->8
        self.ffag4 = FFAGModule(enc_chs, 256, spatial_size=8)
        self.dec4 = ConvBlock3D(256 + 256, 256)

        self.up3 = nn.ConvTranspose3d(256, 128, 2, stride=2)      # 8->16
        self.ffag3 = FFAGModule(enc_chs, 128, spatial_size=16)
        self.dec3 = ConvBlock3D(128 + 128, 128)

        self.up2 = nn.ConvTranspose3d(128, 64, 2, stride=2)       # 16->32
        self.ffag2 = FFAGModule(enc_chs, 64, spatial_size=32)
        self.dec2 = ConvBlock3D(64 + 64, 64)

        self.up1 = nn.ConvTranspose3d(64, 32, 2, stride=2)        # 32->64
        self.ffag1 = FFAGModule(enc_chs, 32, spatial_size=64)
        self.dec1 = ConvBlock3D(32 + 32, 32)

        # Final classification head
        self.final = nn.Conv3d(32, 1, 1)

    def forward(self, enc_feats, bottleneck):
        # Level 4: 4 -> 8
        x = self.up4(bottleneck)
        ffag = self.ffag4(enc_feats)
        x = self.dec4(torch.cat([x, ffag], 1))

        # Level 3: 8 -> 16
        x = self.up3(x)
        ffag = self.ffag3(enc_feats)
        x = self.dec3(torch.cat([x, ffag], 1))

        # Level 2: 16 -> 32
        x = self.up2(x)
        ffag = self.ffag2(enc_feats)
        x = self.dec2(torch.cat([x, ffag], 1))

        # Level 1: 32 -> 64
        x = self.up1(x)
        ffag = self.ffag1(enc_feats)
        x = self.dec1(torch.cat([x, ffag], 1))

        return self.final(x).squeeze(1)  # (B, 64, 64, 64) logits


class BX2SNet(nn.Module):
    """
    Full BX2S-Net: 3D encoder + FFAG decoder.
    Input: (B, 2, 64, 64, 64) dimensionality-enhanced pseudo-3D
    Output: (B, 64, 64, 64) occupancy probability in [0,1]
    """
    def __init__(self):
        super().__init__()
        self.encoder = BX2SEncoder()
        self.decoder = BX2SDecoder()

    def forward(self, x):
        enc_feats, bottleneck = self.encoder(x)
        logits = self.decoder(enc_feats, bottleneck)
        return torch.sigmoid(logits)  # occupancy probabilities


def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

_m = BX2SNet(); print(f"BX2S-Net params: {count_params(_m)/1e6:.1f} M"); del _m


BX2S-Net params: 23.2 M


In [4]:
# ==============================================================================
# LOSS FUNCTIONS — BX2S-Net original (Chen et al., 2023)
# ==============================================================================
# 1. Spatially-weighted Binary Cross-Entropy (boundary voxels weighted higher)
# 2. Dice loss for volumetric overlap
# Total: L = lambda_bce * SW-BCE + lambda_dice * Dice


def spatially_weighted_bce(pred, gt, weight_map):
    """
    Spatially-weighted BCE loss (BX2S-Net paper: edge-enhanced training).
    Boundary voxels get higher weight to improve surface reconstruction.
    """
    eps = 1e-7
    pred = pred.clamp(eps, 1.0 - eps)
    bce = -(gt * torch.log(pred) + (1.0 - gt) * torch.log(1.0 - pred))
    weighted = bce * weight_map
    return weighted.mean()


def dice_loss(pred, gt, smooth=1.0):
    """Dice loss for volumetric overlap."""
    pred_flat = pred.reshape(pred.shape[0], -1)
    gt_flat   = gt.reshape(gt.shape[0], -1)
    intersection = (pred_flat * gt_flat).sum(dim=1)
    union = pred_flat.sum(dim=1) + gt_flat.sum(dim=1)
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return (1.0 - dice).mean()


print("BX2S-Net loss functions defined:")
print(f"  1. Spatially-weighted BCE  (w = {LAMBDA_BCE}, boundary weight = {BOUNDARY_WEIGHT})")
print(f"  2. Dice loss               (w = {LAMBDA_DICE})")


BX2S-Net loss functions defined:
  1. Spatially-weighted BCE  (w = 1.0, boundary weight = 5.0)
  2. Dice loss               (w = 1.0)


In [5]:
# ==============================================================================
# TRAINING LOOP — BX2S-Net: Adam, spatially-weighted BCE + Dice
# ==============================================================================

def collate_fn(b):
    out = {}
    for k in b[0]:
        vals = [x[k] for x in b]
        out[k] = torch.stack(vals, 0) if isinstance(vals[0], torch.Tensor) else vals
    return out

def to_dev(b):
    return {k: (v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v)
            for k, v in b.items()}

def chamfer_np(pred, gt):
    """Evaluation metrics — same as PPCNet v12.3."""
    pred, gt = np.asarray(pred, np.float32), np.asarray(gt, np.float32)
    d = np.linalg.norm(pred[:, None] - gt[None], axis=-1)
    dp, dg = d.min(1), d.min(0)
    def fs(th): p, r = (dp<=th).mean(), (dg<=th).mean(); return 2*p*r/(p+r) if p+r>0 else 0.
    return {"chamfer_mm": float(0.5*(dp.mean()+dg.mean())),
            "fscore_1mm": float(fs(1)), "fscore_2mm": float(fs(2)),
            "fscore_5mm": float(fs(5)),
            "hausdorff_95": float(max(np.percentile(dp,95), np.percentile(dg,95)))}


# -- Build model --
model = BX2SNet().to(device)
print(f"BX2S-Net: {count_params(model)/1e6:.1f} M params")

# BX2S-Net optimizer: Adam, lr=1e-3, weight_decay=1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

train_ds     = LumbarDataset(train_ids, aug=True)
val_ds       = LumbarDataset(val_ids,   aug=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)

print(f"Train : {len(train_ds)} samples -> {len(train_loader)} batches/ep")
print(f"Val   : {len(val_ds)} samples -> {len(val_loader)} batches/ep")


def save_ckpt(path, ep, best, hist):
    tmp = path.with_suffix(".tmp")
    torch.save({"model": model.state_dict(),
                "opt": optimizer.state_dict(),
                "sched": scheduler.state_dict(),
                "epoch": ep, "best_chamfer": best, "history": hist,
                "config": {"method": "BX2S-Net", "paper": "Chen2023"}}, tmp)
    tmp.replace(path)
    print(f"  Saved -> {path.name}  (ep {ep+1})")


def load_ckpt(path):
    st = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(st["model"])
    optimizer.load_state_dict(st["opt"])
    try: scheduler.load_state_dict(st["sched"])
    except: print("  (scheduler state mismatch)")
    ep = st["epoch"] + 1; bc = st.get("best_chamfer", float("inf")); h = st.get("history", [])
    print(f"  Resumed from ep {ep} | best={bc:.3f} mm")
    return ep, bc, h


start_epoch, best_chamfer, history, no_improve = 0, float("inf"), [], 0

if CKPT_PATH.exists():
    print(f"Found checkpoint: {CKPT_PATH.name}")
    start_epoch, best_chamfer, history = load_ckpt(CKPT_PATH)
else:
    print("No checkpoint -- starting fresh.")

print(f"{'='*65}")
print(f"BX2S-Net training from ep {start_epoch+1}/{EPOCHS}")
print(f"Loss: w_bce={LAMBDA_BCE}  w_dice={LAMBDA_DICE}  boundary_w={BOUNDARY_WEIGHT}")
print(f"LR={LR}, cosine annealing")
print(f"{'='*65}")


for epoch in range(start_epoch, EPOCHS):
    model.train()
    t0 = time.time()
    acc_loss, nb = {}, 0

    pbar = tqdm(enumerate(train_loader, 1), total=len(train_loader),
                desc=f'Ep {epoch+1:3d}/{EPOCHS}', leave=True, ncols=130)

    for bi, batch in pbar:
        batch = to_dev(batch)
        input_3d = batch["input_3d"]       # (B, 2, 64, 64, 64)
        gt_vol   = batch["gt_volume"]      # (B, 64, 64, 64)
        bw_map   = batch["boundary_wt"]    # (B, 64, 64, 64)

        optimizer.zero_grad(set_to_none=True)
        pred_vol = model(input_3d)          # (B, 64, 64, 64)

        loss_bce  = LAMBDA_BCE * spatially_weighted_bce(pred_vol, gt_vol, bw_map)
        loss_dice = LAMBDA_DICE * dice_loss(pred_vol, gt_vol)
        loss = loss_bce + loss_dice

        if torch.isfinite(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        acc_loss["bce"]   = acc_loss.get("bce", 0)   + float(loss_bce.detach())
        acc_loss["dice"]  = acc_loss.get("dice", 0)   + float(loss_dice.detach())
        acc_loss["total"] = acc_loss.get("total", 0) + float(loss.detach())
        nb += 1

        if bi % 50 == 0 or bi == len(train_loader):
            g = lambda k: acc_loss.get(k, 0) / max(1, nb)
            pbar.set_postfix_str(
                f"L={g('total'):.3f} bce={g('bce'):.3f} dice={g('dice'):.3f}")

    scheduler.step()
    elapsed = time.time() - t0

    # -- Validation --
    model.eval(); metrics = []; nd = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='  Val', leave=False, ncols=100):
            if nd >= MAX_EVAL: break
            batch = to_dev(batch)
            pred_vol = model(batch["input_3d"])

            pred_vol_np = pred_vol.cpu().numpy()
            gw_np = batch["gt_ppc_world"].cpu().numpy()
            centers = batch["center"].cpu().numpy()
            scales  = batch["scale"].cpu().numpy()

            for b in range(pred_vol_np.shape[0]):
                if nd >= MAX_EVAL: break
                pred_local = volume_to_points(pred_vol_np[b], threshold=0.5,
                                              n_points=N_POINTS)
                pred_world = pred_local * scales[b:b+1] + centers[b:b+1]
                metrics.append(chamfer_np(pred_world, gw_np[b]))
                nd += 1

    if metrics:
        mc = np.mean([m["chamfer_mm"]   for m in metrics])
        f2 = np.mean([m["fscore_2mm"]   for m in metrics])
        hd = np.mean([m["hausdorff_95"] for m in metrics])
        g  = lambda k: acc_loss.get(k, 0) / max(1, nb)
        lr_now = scheduler.get_last_lr()[0]

        print(f"\n[Ep {epoch+1}] {elapsed:.0f}s  lr={lr_now:.2e}  "
              f"Val: Chamfer={mc:.3f}mm  F@2mm={f2:.3f}  HD95={hd:.3f}mm")
        print(f"  Train: bce={g('bce'):.3f} dice={g('dice'):.3f}")

        rec = {"epoch": epoch+1, "chamfer_mm": mc, "f2": f2, "hd95": hd,
               "train_bce": g("bce"), "train_dice": g("dice"), "lr": lr_now}
        history.append(rec); append_log(TRAINING_LOG, rec)
        save_ckpt(CKPT_PATH, epoch, best_chamfer, history)

        if mc < best_chamfer:
            best_chamfer = mc; no_improve = 0
            save_ckpt(BEST_CKPT, epoch, best_chamfer, history)
            print(f"  New best: {best_chamfer:.3f} mm")
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PATIENCE:
                print(f"  Early stop: {no_improve} epochs without improvement"); break

print(f"Done. Best val Chamfer: {best_chamfer:.3f} mm")


BX2S-Net: 23.2 M params
Train : 829 samples -> 208 batches/ep
Val   : 103 samples -> 26 batches/ep
Found checkpoint: latest_checkpoint.pth
  Resumed from ep 75 | best=2.347 mm
BX2S-Net training from ep 76/200
Loss: w_bce=1.0  w_dice=1.0  boundary_w=5.0
LR=0.001, cosine annealing


Ep  76/200:   0%|                                                                                         | 0/208 [00:00<?, ?it/s]/tmp/ipykernel_3033874/1891901083.py:23: DeprecationWarning: BILINEAR is deprecated and will be removed in Pillow 10 (2023-07-01). Use Resampling.BILINEAR instead.
  if img.size != (size, size): img = img.resize((size, size), Image.BILINEAR)
/tmp/ipykernel_3033874/1891901083.py:23: DeprecationWarning: BILINEAR is deprecated and will be removed in Pillow 10 (2023-07-01). Use Resampling.BILINEAR instead.
  if img.size != (size, size): img = img.resize((size, size), Image.BILINEAR)
/tmp/ipykernel_3033874/1891901083.py:23: DeprecationWarning: BILINEAR is deprecated and will be removed in Pillow 10 (2023-07-01). Use Resampling.BILINEAR instead.
  if img.size != (size, size): img = img.resize((size, size), Image.BILINEAR)
/tmp/ipykernel_3033874/1891901083.py:23: DeprecationWarning: BILINEAR is deprecated and will be removed in Pillow 10 (2023-07-01). Use Resamplin


[Ep 76] 83s  lr=6.84e-04  Val: Chamfer=2.397mm  F@2mm=0.443  HD95=6.141mm
  Train: bce=0.223 dice=0.171
  Saved -> latest_checkpoint.pth  (ep 76)


Ep  77/200: 100%|█████████████████████████████████████████████████| 208/208 [00:55<00:00,  3.78it/s, L=0.389 bce=0.221 dice=0.169]



[Ep 77] 55s  lr=6.77e-04  Val: Chamfer=2.390mm  F@2mm=0.444  HD95=6.095mm
  Train: bce=0.221 dice=0.169
  Saved -> latest_checkpoint.pth  (ep 77)


...


[Ep 133] 69s  lr=2.53e-04  Val: Chamfer=2.404mm  F@2mm=0.445  HD95=6.215mm
  Train: bce=0.193 dice=0.136
  Saved -> latest_checkpoint.pth  (ep 133)


Ep 134/200: 100%|█████████████████████████████████████████████████| 208/208 [00:37<00:00,  5.54it/s, L=0.331 bce=0.194 dice=0.137]



[Ep 134] 38s  lr=2.46e-04  Val: Chamfer=2.391mm  F@2mm=0.447  HD95=6.162mm
  Train: bce=0.194 dice=0.137
  Saved -> latest_checkpoint.pth  (ep 134)


Ep 135/200: 100%|█████████████████████████████████████████████████| 208/208 [01:09<00:00,  2.98it/s, L=0.328 bce=0.193 dice=0.135]



[Ep 135] 70s  lr=2.40e-04  Val: Chamfer=2.387mm  F@2mm=0.447  HD95=6.137mm
  Train: bce=0.193 dice=0.135
  Saved -> latest_checkpoint.pth  (ep 135)
  Early stop: 60 epochs without improvement
Done. Best val Chamfer: 2.347 mm


In [ ]:
# ==============================================================================
# TEST EVALUATION — BX2S-Net: volume -> point cloud -> metrics
# ==============================================================================
import gc, torch

# -- Free ALL GPU memory left over from training ------------------------------
try: del optimizer, scheduler
except NameError: pass
try: del model                        # critical — free training model from GPU
except NameError: pass

torch.cuda.synchronize()
torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()             # second pass after gc frees dangling refs

if torch.cuda.is_available():
    free  = torch.cuda.mem_get_info(0)[0] / 1e9
    total = torch.cuda.mem_get_info(0)[1] / 1e9
    print(f"GPU memory free before test load: {free:.2f} / {total:.2f} GB")

# -- Rebuild model fresh, load checkpoint safely via CPU ----------------------
print("Test evaluation...")
if BEST_CKPT.exists():
    st = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)  # CPU first
    model = BX2SNet().cpu()
    model.load_state_dict(st["model"])
    print(f"  Loaded best checkpoint from ep {st['epoch']+1} "
          f"(val Chamfer {st['best_chamfer']:.3f} mm)")
    del st                            # free checkpoint dict before moving to GPU
    gc.collect()
    model = model.to(device)
else:
    print("  WARNING: No best checkpoint -- using current model state.")
    model = BX2SNet().to(device)

if torch.cuda.is_available():
    free = torch.cuda.mem_get_info(0)[0] / 1e9
    print(f"GPU memory free after model load:  {free:.2f} / {total:.2f} GB")

model.eval()

# -- Test DataLoader ----------------------------------------------------------
test_ds     = LumbarDataset(test_ids, aug=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         collate_fn=collate_fn)

# -- Inference loop -----------------------------------------------------------
all_metrics = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='  Test', ncols=100):
        batch = to_dev(batch)
        B = batch["input_3d"].shape[0]

        pred_vol    = model(batch["input_3d"])
        pred_vol_np = pred_vol.cpu().numpy()
        gw_np       = batch["gt_ppc_world"].cpu().numpy()
        centers     = batch["center"].cpu().numpy()
        scales      = batch["scale"].cpu().numpy()
        pids        = batch["patient_id"]

        for b in range(B):
            pred_local = volume_to_points(pred_vol_np[b], threshold=0.5,
                                          n_points=N_POINTS)
            pred_world = pred_local * scales[b:b+1] + centers[b:b+1]
            m = chamfer_np(pred_world, gw_np[b])
            m["patient_id"] = pids[b]
            all_metrics.append(m)
            save_vtk_points(pred_world, RESULTS_DIR / f"{pids[b]}_pred.vtk")

# -- Print results ------------------------------------------------------------
print(f"\n{'='*65}")
print(f"TEST RESULTS -- BX2S-Net ({len(all_metrics)} patients)")
print(f"{'='*65}")
for key, lbl in [("chamfer_mm",  "Chamfer mm  "),
                 ("fscore_1mm",  "F-score @1mm"),
                 ("fscore_2mm",  "F-score @2mm"),
                 ("fscore_5mm",  "F-score @5mm"),
                 ("hausdorff_95","HD95 mm     ")]:
    vals = [m[key] for m in all_metrics]
    print(f"  {lbl}  mean={np.mean(vals):.3f}  std={np.std(vals):.3f}  "
          f"median={np.median(vals):.3f}  p95={np.percentile(vals,95):.3f}")

# -- Save CSV -----------------------------------------------------------------
csv_path = RESULTS_DIR / "test_results_bx2snet.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["patient_id", "chamfer_mm",
                                      "fscore_1mm", "fscore_2mm",
                                      "fscore_5mm", "hausdorff_95"])
    w.writeheader()
    w.writerows(all_metrics)
print(f"\nCSV -> {csv_path}")


In [ ]:
# ==============================================================================
# INTER-COMPARISON TABLE — read all available CSVs
# ==============================================================================

import pandas as pd

results = {}

# BX2S-Net
bx2s_csv = RESULTS_DIR / "test_results_bx2snet.csv"
if bx2s_csv.exists():
    df = pd.read_csv(bx2s_csv)
    results["BX2S-Net"] = {
        "Chamfer": f"{df['chamfer_mm'].mean():.3f} +/- {df['chamfer_mm'].std():.3f}",
        "F@1mm": f"{df['fscore_1mm'].mean():.3f}",
        "F@2mm": f"{df['fscore_2mm'].mean():.3f}",
        "F@5mm": f"{df['fscore_5mm'].mean():.3f}",
        "HD95":  f"{df['hausdorff_95'].mean():.3f} +/- {df['hausdorff_95'].std():.3f}",
    }
    print("BX2S-Net results loaded.")

# PPCNet v12.3
v123_csv = Path("/path/to/inter_comparison_ppcnet_v12_3/results/test_results_v12_3_tta.csv")
if v123_csv.exists():
    df = pd.read_csv(v123_csv)
    results["PPCNet v12.3 (Ours)"] = {
        "Chamfer": f"{df['chamfer_mm'].mean():.3f} +/- {df['chamfer_mm'].std():.3f}",
        "F@1mm": f"{df['fscore_1mm'].mean():.3f}",
        "F@2mm": f"{df['fscore_2mm'].mean():.3f}",
        "F@5mm": f"{df['fscore_5mm'].mean():.3f}",
        "HD95":  f"{df['hausdorff_95'].mean():.3f} +/- {df['hausdorff_95'].std():.3f}",
    }
    print("PPCNet v12.3 results loaded.")

# 3D-ReVert
revert_csv = Path("/path/to/inter_comparison_3drevert/results/test_results_3drevert.csv")
if revert_csv.exists():
    df = pd.read_csv(revert_csv)
    results["3D-ReVert"] = {
        "Chamfer": f"{df['chamfer_mm'].mean():.3f} +/- {df['chamfer_mm'].std():.3f}",
        "F@1mm": f"{df['fscore_1mm'].mean():.3f}",
        "F@2mm": f"{df['fscore_2mm'].mean():.3f}",
        "F@5mm": f"{df['fscore_5mm'].mean():.3f}",
        "HD95":  f"{df['hausdorff_95'].mean():.3f} +/- {df['hausdorff_95'].std():.3f}",
    }
    print("3D-ReVert results loaded.")

# X2CT-GAN
x2ct_csv = Path("/path/to/inter_comparison_x2ctgan/results/test_results_x2ctgan.csv")
if x2ct_csv.exists():
    df = pd.read_csv(x2ct_csv)
    results["X2CT-GAN"] = {
        "Chamfer": f"{df['chamfer_mm'].mean():.3f} +/- {df['chamfer_mm'].std():.3f}",
        "F@1mm": f"{df['fscore_1mm'].mean():.3f}",
        "F@2mm": f"{df['fscore_2mm'].mean():.3f}",
        "F@5mm": f"{df['fscore_5mm'].mean():.3f}",
        "HD95":  f"{df['hausdorff_95'].mean():.3f} +/- {df['hausdorff_95'].std():.3f}",
    }
    print("X2CT-GAN results loaded.")

if results:
    print(f"\n{'='*85}")
    print("INTER-COMPARISON TABLE")
    print(f"{'='*85}")
    header = f"{'Method':<25} {'Chamfer (mm)':<20} {'F@1mm':<8} {'F@2mm':<8} {'F@5mm':<8} {'HD95 (mm)':<20}"
    print(header)
    print("-" * len(header))
    for method, vals in results.items():
        print(f"{method:<25} {vals['Chamfer']:<20} {vals['F@1mm']:<8} "
              f"{vals['F@2mm']:<8} {vals['F@5mm']:<8} {vals['HD95']:<20}")
else:
    print("No results CSVs found yet -- run training first.")


BX2S-Net results loaded.
PPCNet v12.3 results loaded.
X2CT-GAN results loaded.

INTER-COMPARISON TABLE
Method                    Chamfer (mm)         F@1mm    F@2mm    F@5mm    HD95 (mm)           
----------------------------------------------------------------------------------------------
BX2S-Net                  2.564 +/- 1.478      0.081    0.448    0.927    6.722 +/- 5.857     
PPCNet v12.3 (Ours)       1.981 +/- 1.065      0.155    0.646    0.973    4.525 +/- 5.407     
X2CT-GAN                  2.489 +/- 0.928      0.079    0.444    0.933    6.624 +/- 4.673     
